### K-Nearest Neighbors (KNN) Classification
1. Initialize K value. K is the number of nearest neighbors to consider and it controls the balance between overfitting and underfitting. [1 point]

2. Prepare the training data. [1 point]

3.  For each example in the test set

    3.1 Calculate the Euclidean distance between the training set examples (X_train) and the current example from the test data set. [1 point]

    3.2 Sort the distances in ascending order and get the K nearest neighbors based on the calculated distances. [1 point]

    3.3 Get the labels of the K nearest neighbors. [1 point]

    3.4 Get the most common label using [np.unique with return_counts=True and np.argmax] or scipy.stats.mode. [2 points]

    3.5 Append the predicted label to the output list. [1 point]

4. Vectorized prediction (using NumPy operations)

    4.1 Compute the Euclidean distance between all test samples and all training samples at once using NumPy broadcasting without using any loops. This creates a distance matrix of shape (n_test, n_train). [1 point]

    4.2 For each test sample, identify the indices of the K nearest neighbors by sorting the distances along each row. [1 point]

    4.3 Gather the labels of the K nearest neighbors for all test samples. [1 point]

    4.4 Count how many neighbors belong to each class for each test sample using NumPy’s indexed accumulation (np.add.at) to avoid using Python loops. [1 point]

    4.5 Assign each test sample the majority class based on the counts obtained in step 4.4. [1 point]

    4.5 Make sure that the results of the loop and vectorized versions are the same. [1 point]

4. Verify your classifier and fine tune it (change K values to see the change in accuracy) using the Breast Cancer dataset. [1 point for fine-tuning and discussion, and 1 point for successful running of the model]

In [17]:
import numpy as np

class KNNClassifier:
    def __init__(self, k=3):
        # 1. Initialize the number of neighbors K. [1 point]
        self.k = k

    def fit(self, X_train, y_train):
        # 2. Prepare the training data [1 point]
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

    def predict(self, X_test):
        # batch prediction
        predictions = []

        # 3. loop over all samples in the test set
        for sample in X_test:
            
            # 3.1 Compute the distance between the test sample and all training samples in X-train, use np.linalg.norm  [1 point]
            distances = np.linalg.norm(self.X_train - sample, axis=1)

            # 3.2 Sort the distances and return the indices of K nearest neighbors using np.argsort [1 point]
            nearest_neighbors_indices = np.argsort(distances)[:self.k]

            # 3.3 Get the labels of the K nearest neighbors [1 point]
            nearest_labels = self.y_train[nearest_neighbors_indices]
            
            # 3.4 Get the most common label using [np.unique with return_counts=True and np.argmax] or scipy.stats.mode [2 points]
            vals, counts = np.unique(nearest_labels, return_counts=True)
            predicted_label = vals[np.argmax(counts)]
            
            # 3.5 Append the predicted label to the output [1 point]
            predictions.append(predicted_label)

        #return the predictions of all test samples
        return np.array(predictions)
    
    def pred_vec(self, X_test):
        # 4. create a vectorized version of the predict method
        
        # 4.1 Compute pairwise distances at once using NumPy's broadcasting [1 point]
        distances = np.linalg.norm(X_test[:, np.newaxis] - self.X_train, axis=2)

        # 4.2 Sort the distances and return the indices of K nearest neighbors using np.argsort [1 point]
        nearest_indices = np.argsort(distances, axis=1)[:, :self.k]

        # 4.3 Get the labels of the K nearest neighbors [1 point]
        nearest_labels = self.y_train[nearest_indices]

        # 4.4 Count occurrences for each class using np.add.at [1 point]
        n_test = len(X_test)
        n_classes = np.max(self.y_train) + 1

        n_test = X_test.shape[0]
        n_classes = int(np.max(self.y_train) + 1)
        counts = np.zeros((n_test, n_classes))
        
        rows = np.repeat(np.arange(n_test), self.k)
        
        labels_flat = nearest_labels.flatten()
        np.add.at(counts, (rows, labels_flat), 1)

        # 4.5 Obtain the majority vote of all samples [1 point]
        predictions = np.argmax(counts, axis=1)

        #return the predictions of all test samples
        return predictions

In [26]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

# Load dataset
breast_cancer = load_breast_cancer()
X = breast_cancer.data
y = breast_cancer.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train KNN classifier
knn_classifier = KNNClassifier(k=20)
knn_classifier.fit(X_train, y_train)

# Make predictions

# Loop version: 
start_time = time.time()
predictions_loop = knn_classifier.predict(X_test)
end_time = time.time()

# Calculate accuracy [1 point for good accuracy]
accuracy_loop = accuracy_score(y_test, predictions_loop)
time_loop = end_time - start_time
print(f"Standard predict accuracy: {accuracy_loop:.4f}, time: {time_loop:.4f} sec")

# Vectorized version:
start_time = time.time()
predictions_vec = knn_classifier.pred_vec(X_test)
end_time = time.time()

# Calculate accuracy [should be the same as the loop version]
accuracy_vec = accuracy_score(y_test, predictions_vec)
time_vec = end_time - start_time
print(f"Vectorized pred_vec accuracy: {accuracy_vec:.4f}, time: {time_vec:.4f} sec")

# Check that the predictions of the vectorized and non-vectorized versions are the same
print(f"The results between the non-vectorized and vectorized versions are the same: {np.all(predictions_loop == predictions_vec)}")

# Speed-up
# Note: Speed-up is not guaranteed here since the dataset is small.
# On larger datasets, the vectorized pred_vec() can be much faster.
speedup = time_loop / time_vec
print(f"Speed-up: {speedup:.2f}x")

# Remember you need to fine tune your classifier (change K values to see the change in accuracy). 
# 1 point will be given for fine-tuning and a brief discussion.

Standard predict accuracy: 0.9649, time: 0.0091 sec
Vectorized pred_vec accuracy: 0.9649, time: 0.0098 sec
The results between the non-vectorized and vectorized versions are the same: True
Speed-up: 0.93x


Below is a summary of the tested K values.

| K | Accuracy |
| :--- | :--- |
| 1 | 0.9298 |
| 2 | 0.9298 |
| 3 | 0.9298 |
| 5 | 0.9561 |
| 10 | 0.9737 |
| 15 | 0.9649 |
| 20 | 0.9649 |

Of the testesd values, K=10 gives the best model accuracy. Values less than 10 overfit and values over 10 underfit.